# **Aula 2**: Extração e Ingestão de Dados

OBJETIVOS:

- Explorar o ecossistema Pandas para ingestão de múltiplas fontes.
- Compreender a diferença entre dados estruturados (CSV) e semiestruturados (JSON/API).
- Aprender técnicas de inspeção de "Schema" (tipagem de dados).
- Realizar os primeiros tratamentos de limpeza pós-extração.

In [0]:
import pandas as pd
import numpy as np
import requests

## 1. Ingestão de Dados Estruturados (CSV)
Diferente do Excel, na Ciência de Dados conectamos o Python diretamente à fonte.

Vamos utilizar um dataset real de preços de combustíveis.

In [0]:
# Importando dados de um repositório público (Ex: PIB Mundial por país)
url_pib = "https://raw.githubusercontent.com/datasets/gdp/master/data/gdp.csv"
df_pib = pd.read_csv(url_pib)

print("Ingestão Concluída")
print(f"Dimensões do Dataset: {df_pib.shape[0]} linhas e {df_pib.shape[1]} colunas.")

## 2. Inspeção Técnica e "Schema"
Antes de calcular, precisamos saber se o Python entendeu o que é cada coluna.

Erro comum: Datas ou Números sendo lidos como Texto (object).

In [0]:
print("--- Verificação de Tipagem (Schema) ---")
print(df_pib.dtypes)

print("\n--- Diagnóstico de Valores Ausentes ---")
print(df_pib.isnull().sum())

In [0]:
# Estatística Descritiva (O 'Raio-X' dos dados)
print("\nResumo Estatístico:")
display(df_pib.describe())

## 3. Extração via API (Dados Dinâmicos)
APIs são pontes entre sistemas. Vamos extrair a cotação atual de moedas em formato JSON.

In [0]:
url_api = "https://api.exchangerate-api.com/v4/latest/USD"
response = requests.get(url_api)
data = response.json()

In [0]:
# O dado vem como um dicionário complexo. Precisamos "achatar" para uma tabela.
df_cambio = pd.DataFrame(list(data['rates'].items()), columns=['Moeda', 'Taxa'])
print("Exemplo de dados extraídos via API:")
display(df_cambio.head(10))

In [0]:
# Verificando valores nulos
print("Valores ausentes por coluna:")
print(df_cambio.isnull().sum())

## 4. Manipulação e Transformação Inicial
Vamos criar uma nova coluna baseada em regras lógicas (Vetorização).

Se a taxa for > 5.0, classificar como 'Moeda Desvalorizada'.

In [0]:
df_cambio['Status'] = np.where(df_cambio['Taxa'] > 5.0, 'Desvalorizada', 'Normal')
display(df_cambio.query("Moeda == 'BRL' or Moeda == 'EUR' or Moeda == 'JPY'"))

In [0]:
# Estatística Descritiva (O 'Raio-X' dos dados)
print("\nResumo Estatístico:")
display(df_cambio.describe())

## 5.1 Identificando e tratando valores ausentes (NaN)


In [0]:
print("--- Análise de Missing Data ---")
# Visualizando onde estão os buracos no dataset
missing_count = df_pib.isnull().sum()
print(f"Colunas com valores nulos:\n{missing_count}")

In [0]:
# Estratégia: Preenchimento (Imputação) vs Exclusão
# Aqui, preenchemos valores nulos de 'Value' com a mediana para não distorcer por outliers
df_pib['Value'] = df_pib['Value'].fillna(df_pib['Value'].median())

# Se o Código do País estiver nulo, a linha é inválida para o modelo, então removemos.
df_pib.dropna(subset=['Country Code'], inplace=True)

## 5.2 Correção de Schema (Tipagem)
Muitas vezes o ano é lido como float ou string. Vamos garantir que seja Inteiro.

In [0]:
df_pib['Year'] = df_pib['Year'].astype(int)

## 5.3 Padronização de Texto (Garantindo Consistência)
Transformando nomes de países em maiúsculo para evitar conflitos (ex: 'Brazil' vs 'brazil')

In [0]:
df_pib['Country Name'] = df_pib['Country Name'].str.upper().str.strip()
print("\nDados limpos e padronizados!")

## 6.1 Operação Vetorizada vs Loop
Suponha que queremos converter o valor do PIB para uma escala logarítmica para modelos de ML

In [0]:
import math

# FORMA LENTA (Loop):
for i in range(len(df_pib)): df_pib.iloc[i, 3] = math.log(df_pib.iloc[i, 3])

In [0]:
# FORMA RÁPIDA (Vetorização)
df_pib['Log_Value'] = np.log1p(df_pib['Value'])

print("Cálculo matemático aplicado a milhões de linhas instantaneamente via Álgebra Linear.")

## 7.1 Processamento em Chunks (Pedaços)

In [0]:
# Se o arquivo fosse de 10GB, leríamos assim:
caminho_csv = "https://raw.githubusercontent.com/datasets/gdp/master/data/gdp.csv"

In [0]:
# Lendo em pedaços de 5000 linhas
chunks = pd.read_csv(caminho_csv, chunksize=5000)

for i, pedaco in enumerate(chunks):
    processado = pedaco['Value'].mean() # Processa e descarta o resto da memória
    print(f"Processado pedaço {i+1}...")
    if i == 2: break # Apenas exemplo

## 7.2 Otimização de Formato: Parquet
O CSV é texto plano. O Parquet é binário e colunar.

In [0]:
!pip install --upgrade pyarrow pandas

**Reinicie o Kernel**: No menu do Jupyter/Colab, vá em Kernel -> Restart (ou Ambiente de Execução -> Reiniciar).

 Isso é necessário para o Python carregar as novas bibliotecas de sistema.

 Lembre de rodar novamente as células anteriores, para importarmos as bibliotecas que estavamos utilizando.

In [0]:
df_pib.to_parquet('pib_otimizado.parquet')
print("Arquivo salvo em Parquet: Leitura até 10x mais rápida que CSV.")

In [0]:
import os

# 1. Exportando os arquivos para comparação
df_pib.to_csv('pib_dados.csv', index=False)
df_pib.to_parquet('pib_dados.parquet', index=False)

# 2. Obtendo o tamanho dos arquivos em Kilobytes (KB)
tamanho_csv = os.path.getsize('pib_dados.csv') / 1024
tamanho_parquet = os.path.getsize('pib_dados.parquet') / 1024

# 3. Calculando a taxa de compressão
economia = 100 * (1 - (tamanho_parquet / tamanho_csv))

print(f"--- Relatório de Armazenamento ---")
print(f"Tamanho do arquivo CSV: {tamanho_csv:.2f} KB")
print(f"Tamanho do arquivo Parquet: {tamanho_parquet:.2f} KB")
print(f"Economia de espaço: {economia:.1f}%")

if tamanho_parquet < tamanho_csv:
    print("\nConclusão: O Parquet é menor devido à compressão colunar e tipagem binária.")

In [0]:
# No CSV, o pandas lê o arquivo todo. No Parquet, podemos ler apenas colunas específicas.
# Isso é essencial quando lidamos com Big Data.

# Lendo apenas as colunas 'Year' e 'Value' do arquivo Parquet que criamos
df_reduzido = pd.read_parquet('pib_dados.parquet', columns=['Year', 'Value'])

print("--- Leitura Otimizada (Somente colunas necessárias) ---")
display(df_reduzido.head())

# Verificando que as outras colunas (como 'Country Name') não ocupam espaço na RAM agora
print(f"Colunas na memória: {df_reduzido.columns.tolist()}")

## 8.1 Anonimização de Dados Sensíveis
Em Ciência de Dados, muitas vezes precisamos remover IDs ou nomes reais

In [0]:
def anonimizar(nome):
    return "USER_" + str(hash(nome))[:8]

# Criando uma coluna anonimizada para simular conformidade com LGPD
df_pib['ID_Anonimo'] = df_pib['Country Code'].apply(anonimizar)
display(df_pib[['Country Name', 'ID_Anonimo']].head())

In [0]:
import hashlib

# Função para anonimizar nomes de países ou IDs de usuários
def anonimizar_dado(texto):
    return hashlib.sha256(texto.encode()).hexdigest()[:10]

# Criando uma coluna de ID Protegido para o Data Lake (ELT)
df_pib['ID_Protegido'] = df_pib['Country Code'].apply(anonimizar_dado)

print("Exemplo de dado anonimizado para conformidade com a LGPD:")
display(df_pib[['Country Name', 'ID_Protegido']].head())

## Simulando erro de extração comum:
## Encoding (Latin-1 vs UTF-8)
Essencial para garantir a integridade no transporte

In [0]:
url_vendas = "https://raw.githubusercontent.com/fivethirtyeight/data/master/comic-characters/dc-wikia-data.csv"

try:
    # Tentativa de leitura de um arquivo com delimitador e encoding específicos
    df_vendas = pd.read_csv(url_vendas, sep=',', encoding='utf-8')
    print("Dica: Use 'sep' e 'encoding' para arquivos que não seguem o padrão global.")
    display(df_vendas.head(3))
except Exception as e:
    print(f"Erro de Ingestão: {e}")

In [0]:
# Salvando os dados reais carregados (DC Characters) nos dois formatos
df_vendas.to_csv('personagens.csv', index=False)
df_vendas.to_parquet('personagens.parquet', index=False)

# Medindo os tamanhos
t_csv = os.path.getsize('personagens.csv') / 1024
t_parquet = os.path.getsize('personagens.parquet') / 1024

print(f"--- Prova de Otimização ---")
print(f"CSV: {t_csv:.2f} KB")
print(f"Parquet: {t_parquet:.2f} KB")
print(f"Economia de Espaço: {100 * (1 - t_parquet/t_csv):.1f}%")